# GPU code-block test, 80 GB profile (Colab A100/H100 80 GB)

Runs the 4 Track 08 blocks whose defaults assume an 80 GB GPU **with the chapter defaults** (RMM pool 60GB/76GB and 64GB,
memory-guard thresholds 70/65 GB) on synthetic data of 25k and 100k cells x 35,000 genes. Each block runs in its own process and
the peak GPU memory (nvidia-smi polling, includes ~0.4 GB of the parent CUDA context) and the peak host RSS are recorded in `gpu_results_80gb.json`.
The ramp stops at 100k cells on purpose (host-RAM risk of a 1M x 35k synthetic atlas was not taken).

## 0. GPU check (refuses to run below 77,000,000,000 bytes)

In [ ]:
import subprocess, os, torch
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "NO GPU DETECTED"
props = torch.cuda.get_device_properties(0)
GPU_NAME, GPU_MEM_BYTES = props.name, props.total_memory
print(f"GPU: {GPU_NAME} | total_memory = {GPU_MEM_BYTES} bytes ({GPU_MEM_BYTES/1024**3:.1f} GiB)")
if GPU_MEM_BYTES < 77000000000:
    raise RuntimeError(f"GPU has only {GPU_MEM_BYTES/1e9:.1f} GB; the chapter default RMM max pool is 76GB (decimal). Stopping before the installs.")

## 1. Install (RAPIDS only; scvi-tools / cell2location are not needed for these blocks)

In [ ]:
import re, subprocess
smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout
m = re.search(r"CUDA Version:\s*(\d+)\.", smi)
cuda_major = int(m.group(1)) if m else 12
extra = "rapids-cu13" if cuda_major >= 13 else "rapids-cu12"
print("driver CUDA major:", cuda_major, "-> extra:", extra)
!pip install -q "rapids-singlecell[{extra}]" psutil --extra-index-url=https://pypi.nvidia.com 2>&1 | tail -5

In [ ]:
import importlib, os, psutil
VERSIONS = {}
for name in ["torch", "scanpy", "anndata", "cupy", "cudf", "cuml", "cugraph", "cuvs", "rmm", "rapids_singlecell"]:
    try:
        VERSIONS[name] = getattr(importlib.import_module(name), "__version__", "unknown")
    except Exception as e:
        VERSIONS[name] = f"IMPORT FAILED: {type(e).__name__}: {str(e)[:150]}"
HOST = {"cpus": os.cpu_count(), "ram_gb": round(psutil.virtual_memory().total / 1e9, 1)}
print(VERSIONS); print(HOST)

## 2. Harness: every block in its own subprocess, with peak GPU / host memory

In [ ]:
import json, re, subprocess, sys, time
RESULTS = {"meta": {}, "blocks": {}}
CHILD = r'''
import json, sys, time, threading, subprocess, traceback, resource, warnings
payload = json.load(open(sys.argv[1]))
bid, loc = payload["bid"], payload["loc"]
rec = {"id": bid, "location": loc, "status": None, "error": None, "traceback": None}
peak = {"gpu_mib": 0}
stop = threading.Event()
def poll():
    while not stop.is_set():
        try:
            out = subprocess.run(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"],
                                 capture_output=True, text=True, timeout=5).stdout.split()[0]
            peak["gpu_mib"] = max(peak["gpu_mib"], int(out))
        except Exception:
            pass
        stop.wait(0.5)
threading.Thread(target=poll, daemon=True).start()
ns = {"__name__": "__main__"}
t0 = time.time()
with warnings.catch_warnings(record=True) as wl:
    warnings.simplefilter("always")
    try:
        exec(compile(payload["block"], "<%s %s>" % (bid, loc), "exec"), ns)
        exec(compile(payload["post"], "<%s HARNESS>" % bid, "exec"), {"ns": ns, "__name__": "__main__"})
        rec["status"] = "PASS"
    except BaseException as e:
        rec["status"] = "FAIL"
        rec["error"] = "%s: %s" % (type(e).__name__, str(e)[:600])
        rec["traceback"] = traceback.format_exc()[-2500:]
    rec["warnings"] = sorted({"%s: %s" % (w.category.__name__, str(w.message)[:160]) for w in wl})[:8]
rec["elapsed_s"] = round(time.time() - t0, 2)
rec["peak_gpu_used_mib"] = peak["gpu_mib"]
rec["peak_host_rss_mib"] = round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024)
stop.set()
print("@@RESULT@@" + json.dumps(rec))
'''

def run_block(bid, location, src, post, notes="", timeout=1800, replace=()):
    applied = []
    for a, b in replace:
        if a in src:
            src = src.replace(a, b)
            applied.append([a, b])
    pj = f"/content/_blk_{bid}.json"
    json.dump({"bid": bid, "loc": location, "block": src, "post": post}, open(pj, "w"))
    open("/content/_child.py", "w").write(CHILD)
    try:
        p = subprocess.run([sys.executable, "/content/_child.py", pj], capture_output=True, text=True, timeout=timeout)
        out, err, rc = p.stdout, p.stderr, p.returncode
    except subprocess.TimeoutExpired as e:
        tx = lambda b: b.decode() if isinstance(b, bytes) else (b or "")
        out, err, rc = tx(e.stdout), tx(e.stderr), "TIMEOUT"
    m = re.search(r"@@RESULT@@(.*)", out)
    rec = json.loads(m.group(1)) if m else {"id": bid, "location": location, "status": "CRASH" if rc != "TIMEOUT" else "TIMEOUT",
                                             "error": f"no result line; returncode={rc}", "elapsed_s": None}
    rec["stdout_tail"] = [l for l in out.splitlines() if "@@RESULT@@" not in l][-30:]
    rec["stderr_tail"] = err.splitlines()[-8:]
    rec["notes"] = notes
    rec["replacements"] = applied
    RESULTS["blocks"][bid] = rec
    print(f"[{rec['status']}] {bid} {rec.get('elapsed_s')}s  peak GPU {rec.get('peak_gpu_used_mib')} MiB  peak host RSS {rec.get('peak_host_rss_mib')} MiB"
          + (f"\n    {rec['error']}" if rec.get("error") else ""))
    print("    " + "\n    ".join(rec["stdout_tail"][-14:]))
    return rec

## 3. Synthetic atlases (35,000 genes, ~5% non-zero, first 13 genes named `MT-*`)

In [ ]:
import numpy as np, scipy.sparse as sp, anndata as ad, pandas as pd
G = 35000
def make_atlas(n, path, chunk=2500, seed=1):
    t0 = time.time()
    rng = np.random.default_rng(seed)
    lam = rng.gamma(0.3, 0.2, G)                       # per-gene Poisson rate -> ~5% non-zero entries
    parts = [sp.csr_matrix(rng.poisson(lam, (min(chunk, n - i), G)).astype("float32")) for i in range(0, n, chunk)]
    X = sp.vstack(parts, format="csr")
    names = [f"MT-{j}" if j < 13 else f"G{j}" for j in range(G)]
    ad.AnnData(X, obs=pd.DataFrame(index=[f"c{j}" for j in range(n)]), var=pd.DataFrame(index=names)).write_h5ad(path)
    print(f"{path}: {n} x {G}, nnz={X.nnz:,} ({X.nnz/(n*G):.1%}), {time.time()-t0:.0f}s, host RAM used {psutil.virtual_memory().used/1e9:.1f} GB")
    return X.nnz
SIZES = [25000, 100000]
NNZ = {n: make_atlas(n, f"/content/atlas_{n//1000}k.h5ad") for n in SIZES}

## 4. Blocks (chapter defaults; the HARNESS post code only builds inputs and calls the functions the block defines)

### B4_25k  `tracks/08_large_scale_sparse/docs/07_GPU_ACCELERATION_RAPIDS_SINGLECELL.md:184` (T08 ch07 rapids-singlecell pipeline)
Chapter-default RMM pool 60GB/76GB, synthetic atlas 25,000 x 35,000.

In [ ]:
B4_SRC = r'''"""
End-to-End GPU Single-Cell Acceleration Pipeline with rapids-singlecell
Benchmarked on 1,000,000 Prostate Cancer Single-Cell Atlas
"""

import time
import os
import gc
import numpy as np
import scipy.sparse as sp
import cupy as cp
import cupyx.scipy.sparse as cpsp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import rapids_singlecell as rsc
import scanpy as sc
import anndata as ad


def init_gpu_memory_pool(initial_pool_size: str = "60GB", max_pool_size: str = "76GB"):
    """
    RAPIDS 메모리 매니저(RMM) 풀을 초기화하여 CudaMalloc 오버헤드와 단편화를 원천 차단한다.
    """
    print(f"[RMM Init] Allocating GPU Memory Pool ({initial_pool_size} ~ {max_pool_size})...")
    rmm.reinitialize(
        pool_allocator=True,
        initial_pool_size=initial_pool_size,  # RMM이 "60GB" 형식 문자열을 직접 해석 (GB=10^9, GiB=2^30 bytes)
        maximum_pool_size=max_pool_size
    )
    # CuPy의 기본 메모리 할당자를 RMM으로 바인딩
    cp.cuda.set_allocator(rmm_cupy_allocator)
    print("[RMM Init] GPU Memory Pool successfully bound to CuPy & RAPIDS.")


def load_1m_atlas_to_gpu(h5ad_path: str) -> ad.AnnData:
    """
    1M 전립선암 H5AD 파일을 디스크에서 로드하고 AnnData의 카운트 행렬을 GPU CuPy CSR로 즉각 전송한다.
    """
    t0 = time.time()
    print(f"[I/O] Reading H5AD from {h5ad_path}...")
    adata = sc.read_h5ad(h5ad_path)
    print(f"[I/O] Host read completed in {time.time() - t0:.2f}s. Shape: {adata.shape}")

    t_gpu = time.time()
    print("[PCIe] Promoting sparse count matrix to GPU VRAM (cupyx.scipy.sparse.csr_matrix)...")
    if sp.issparse(adata.X):
        # SciPy CSR을 CuPy CSR로 비동기 복사
        adata.X = cpsp.csr_matrix(adata.X, dtype=cp.float32)
    else:
        adata.X = cp.asarray(adata.X, dtype=cp.float32)

    print(f"[PCIe] Transfer completed in {time.time() - t_gpu:.2f}s.")
    return adata


def run_gpu_accelerated_pipeline(adata: ad.AnnData) -> ad.AnnData:
    """
    1M 전립선암 세포에 대해 전처리부터 UMAP까지 GPU 메모리 내에서 연속 체이닝 실행
    """
    total_t0 = time.time()
    print("\n" + "="*65)
    print("   STARTING RAPIDS-SINGLECELL 1M CELLS GPU PIPELINE")
    print("="*65)

    # 1. GPU 기반 고속 품질 관리 (QC Metrics)
    t0 = time.time()
    # 미토콘드리아 유전자 플래그 등록
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    rsc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], log1p=False)
    print(f"[Stage 1] GPU QC Metrics calculation : {time.time() - t0:.3f}s")

    # 2. 세포 및 유전자 필터링 (In-place Boolean Masking on GPU)
    t0 = time.time()
    rsc.pp.filter_cells(adata, min_genes=200)
    rsc.pp.filter_genes(adata, min_cells=10)
    print(f"[Stage 2] GPU Filter Cells & Genes   : {time.time() - t0:.3f}s (Remaining: {adata.n_obs:,} cells)")

    # 3. 초고속 고변동 유전자(HVG) 선별 (Warp Shuffle Reduction)
    #    flavor="seurat_v3"는 정규화·로그 변환 전의 원시 카운트를 요구한다 (scanpy/rsc 문서)
    t0 = time.time()
    rsc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat_v3")
    print(f"[Stage 3] GPU Highly Variable Genes  : {time.time() - t0:.3f}s")

    # 4. 라이브러리 크기 정규화 및 Log1p 변환 (GPU Parallel Kernel)
    t0 = time.time()
    rsc.pp.normalize_total(adata, target_sum=1e4)
    rsc.pp.log1p(adata)
    print(f"[Stage 4] GPU Normalize & Log1p      : {time.time() - t0:.3f}s")

    # 5. HVG 부분집합 선택 후 유전자별 표준화(Scaling) 및 클리핑
    #    부분집합 없이 전체 35,000개 유전자에 scale을 적용하면 1M x 35k 밀집 행렬(약 140 GB)이 되어 VRAM을 초과한다
    t0 = time.time()
    adata = adata[:, adata.var["highly_variable"]].copy()
    rsc.pp.scale(adata, max_value=10.0)
    print(f"[Stage 5] GPU Matrix Scaling (HVG)   : {time.time() - t0:.3f}s")

    # 6. GPU 주성분 분석 (rsc.pp.pca; 구버전은 rsc.tl.pca)
    t0 = time.time()
    rsc.pp.pca(adata, n_comps=50, zero_center=True)
    print(f"[Stage 6] GPU PCA (50 components)    : {time.time() - t0:.3f}s")

    # 7. GPU 근사 최근접 이웃 탐색 (kNN Graph Construction via cuML)
    t0 = time.time()
    rsc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, metric="euclidean")
    print(f"[Stage 7] GPU kNN Graph (k=15, 1M)   : {time.time() - t0:.3f}s")

    # 8. cuML 병렬 비동기 SGD UMAP 2D 임베딩
    t0 = time.time()
    rsc.tl.umap(adata, min_dist=0.3, spread=1.0)
    print(f"[Stage 8] cuML UMAP 2D Embedding     : {time.time() - t0:.3f}s")

    total_time = time.time() - total_t0
    print("="*65)
    print(f"[SUCCESS] Complete 1M Atlas GPU Pipeline Finished in: {total_time:.2f} seconds!")
    print("="*65 + "\n")

    return adata
'''
POST = '\nns["init_gpu_memory_pool"]()          # chapter defaults: initial 60GB, max 76GB (no override)\na = ns["load_1m_atlas_to_gpu"]("/content/atlas_25k.h5ad")\na = ns["run_gpu_accelerated_pipeline"](a)\nassert "X_umap" in a.obsm\nprint("HARNESS: cells after QC =", a.n_obs, "| genes after filter =", a.n_vars)\n'
run_block('B4_25k', 'tracks/08_large_scale_sparse/docs/07_GPU_ACCELERATION_RAPIDS_SINGLECELL.md:184', B4_SRC, POST, notes='', replace=[])

### B4_100k  `tracks/08_large_scale_sparse/docs/07_GPU_ACCELERATION_RAPIDS_SINGLECELL.md:184` (T08 ch07 rapids-singlecell pipeline)
Chapter-default RMM pool 60GB/76GB, synthetic atlas 100,000 x 35,000.

In [ ]:
B4_SRC = r'''"""
End-to-End GPU Single-Cell Acceleration Pipeline with rapids-singlecell
Benchmarked on 1,000,000 Prostate Cancer Single-Cell Atlas
"""

import time
import os
import gc
import numpy as np
import scipy.sparse as sp
import cupy as cp
import cupyx.scipy.sparse as cpsp
import rmm
from rmm.allocators.cupy import rmm_cupy_allocator
import rapids_singlecell as rsc
import scanpy as sc
import anndata as ad


def init_gpu_memory_pool(initial_pool_size: str = "60GB", max_pool_size: str = "76GB"):
    """
    RAPIDS 메모리 매니저(RMM) 풀을 초기화하여 CudaMalloc 오버헤드와 단편화를 원천 차단한다.
    """
    print(f"[RMM Init] Allocating GPU Memory Pool ({initial_pool_size} ~ {max_pool_size})...")
    rmm.reinitialize(
        pool_allocator=True,
        initial_pool_size=initial_pool_size,  # RMM이 "60GB" 형식 문자열을 직접 해석 (GB=10^9, GiB=2^30 bytes)
        maximum_pool_size=max_pool_size
    )
    # CuPy의 기본 메모리 할당자를 RMM으로 바인딩
    cp.cuda.set_allocator(rmm_cupy_allocator)
    print("[RMM Init] GPU Memory Pool successfully bound to CuPy & RAPIDS.")


def load_1m_atlas_to_gpu(h5ad_path: str) -> ad.AnnData:
    """
    1M 전립선암 H5AD 파일을 디스크에서 로드하고 AnnData의 카운트 행렬을 GPU CuPy CSR로 즉각 전송한다.
    """
    t0 = time.time()
    print(f"[I/O] Reading H5AD from {h5ad_path}...")
    adata = sc.read_h5ad(h5ad_path)
    print(f"[I/O] Host read completed in {time.time() - t0:.2f}s. Shape: {adata.shape}")

    t_gpu = time.time()
    print("[PCIe] Promoting sparse count matrix to GPU VRAM (cupyx.scipy.sparse.csr_matrix)...")
    if sp.issparse(adata.X):
        # SciPy CSR을 CuPy CSR로 비동기 복사
        adata.X = cpsp.csr_matrix(adata.X, dtype=cp.float32)
    else:
        adata.X = cp.asarray(adata.X, dtype=cp.float32)

    print(f"[PCIe] Transfer completed in {time.time() - t_gpu:.2f}s.")
    return adata


def run_gpu_accelerated_pipeline(adata: ad.AnnData) -> ad.AnnData:
    """
    1M 전립선암 세포에 대해 전처리부터 UMAP까지 GPU 메모리 내에서 연속 체이닝 실행
    """
    total_t0 = time.time()
    print("\n" + "="*65)
    print("   STARTING RAPIDS-SINGLECELL 1M CELLS GPU PIPELINE")
    print("="*65)

    # 1. GPU 기반 고속 품질 관리 (QC Metrics)
    t0 = time.time()
    # 미토콘드리아 유전자 플래그 등록
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    rsc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], log1p=False)
    print(f"[Stage 1] GPU QC Metrics calculation : {time.time() - t0:.3f}s")

    # 2. 세포 및 유전자 필터링 (In-place Boolean Masking on GPU)
    t0 = time.time()
    rsc.pp.filter_cells(adata, min_genes=200)
    rsc.pp.filter_genes(adata, min_cells=10)
    print(f"[Stage 2] GPU Filter Cells & Genes   : {time.time() - t0:.3f}s (Remaining: {adata.n_obs:,} cells)")

    # 3. 초고속 고변동 유전자(HVG) 선별 (Warp Shuffle Reduction)
    #    flavor="seurat_v3"는 정규화·로그 변환 전의 원시 카운트를 요구한다 (scanpy/rsc 문서)
    t0 = time.time()
    rsc.pp.highly_variable_genes(adata, n_top_genes=3000, flavor="seurat_v3")
    print(f"[Stage 3] GPU Highly Variable Genes  : {time.time() - t0:.3f}s")

    # 4. 라이브러리 크기 정규화 및 Log1p 변환 (GPU Parallel Kernel)
    t0 = time.time()
    rsc.pp.normalize_total(adata, target_sum=1e4)
    rsc.pp.log1p(adata)
    print(f"[Stage 4] GPU Normalize & Log1p      : {time.time() - t0:.3f}s")

    # 5. HVG 부분집합 선택 후 유전자별 표준화(Scaling) 및 클리핑
    #    부분집합 없이 전체 35,000개 유전자에 scale을 적용하면 1M x 35k 밀집 행렬(약 140 GB)이 되어 VRAM을 초과한다
    t0 = time.time()
    adata = adata[:, adata.var["highly_variable"]].copy()
    rsc.pp.scale(adata, max_value=10.0)
    print(f"[Stage 5] GPU Matrix Scaling (HVG)   : {time.time() - t0:.3f}s")

    # 6. GPU 주성분 분석 (rsc.pp.pca; 구버전은 rsc.tl.pca)
    t0 = time.time()
    rsc.pp.pca(adata, n_comps=50, zero_center=True)
    print(f"[Stage 6] GPU PCA (50 components)    : {time.time() - t0:.3f}s")

    # 7. GPU 근사 최근접 이웃 탐색 (kNN Graph Construction via cuML)
    t0 = time.time()
    rsc.pp.neighbors(adata, n_neighbors=15, n_pcs=50, metric="euclidean")
    print(f"[Stage 7] GPU kNN Graph (k=15, 1M)   : {time.time() - t0:.3f}s")

    # 8. cuML 병렬 비동기 SGD UMAP 2D 임베딩
    t0 = time.time()
    rsc.tl.umap(adata, min_dist=0.3, spread=1.0)
    print(f"[Stage 8] cuML UMAP 2D Embedding     : {time.time() - t0:.3f}s")

    total_time = time.time() - total_t0
    print("="*65)
    print(f"[SUCCESS] Complete 1M Atlas GPU Pipeline Finished in: {total_time:.2f} seconds!")
    print("="*65 + "\n")

    return adata
'''
POST = '\nns["init_gpu_memory_pool"]()          # chapter defaults: initial 60GB, max 76GB (no override)\na = ns["load_1m_atlas_to_gpu"]("/content/atlas_100k.h5ad")\na = ns["run_gpu_accelerated_pipeline"](a)\nassert "X_umap" in a.obsm\nprint("HARNESS: cells after QC =", a.n_obs, "| genes after filter =", a.n_vars)\n'
run_block('B4_100k', 'tracks/08_large_scale_sparse/docs/07_GPU_ACCELERATION_RAPIDS_SINGLECELL.md:184', B4_SRC, POST, notes='', replace=[])

### B5_25k  `tracks/08_large_scale_sparse/docs/08_GPU_GRAPH_COMMUNITY_DETECTION.md:171` (T08 ch08 CAGRA + cuGraph Leiden)
25,000 random 50-d points (the chapter targets 1M).

In [ ]:
B5_SRC = r'''"""
Large-Scale Single-Cell Graph Construction & Community Detection
NVIDIA CAGRA + cuGraph Leiden on 1,000,000 Prostate Cancer Atlas
"""

import time
import numpy as np
import cupy as cp
import cudf
import cugraph
# CAGRA는 RAFT(pylibraft)에서 RAPIDS cuVS로 이전되었다 (cuvs.neighbors.cagra; cuVS 소스의 시그니처 기준, GPU 미실행)
from cuvs.common import Resources
from cuvs.neighbors import cagra
import scipy.sparse as sp
import cupyx.scipy.sparse as cpsp
from typing import Tuple


def build_cagra_knn_graph_1m(X_pca_cp: cp.ndarray, k: int = 15) -> Tuple[cp.ndarray, cp.ndarray]:
    """
    NVIDIA CAGRA를 활용하여 100만 세포(50 PCs)의 kNN 그래프를 GPU 네이티브로 초고속 구축한다.
    
    Parameters:
        X_pca_cp: GPU VRAM에 상주하는 CuPy float32 배열 (1,000,000 x 50)
        k: 탐색할 최근접 이웃 수 (기본 15)
        
    Returns:
        knn_indices: 각 세포별 k개 이웃의 정점 인덱스 (1,000,000 x k, int32)
        knn_distances: 유클리드 거리 행렬 (1,000,000 x k, float32)
    """
    N, d = X_pca_cp.shape
    print(f"\n[CAGRA] Initializing Fixed Out-Degree Graph Index for {N:,} cells (dim={d})...")
    t0 = time.time()
    
    handle = Resources()
    
    # 1. CAGRA 인덱스 빌드 파라미터 구성
    build_params = cagra.IndexParams(
        intermediate_graph_degree=64,   # 중간 탐색 그래프 차수
        graph_degree=32,                # 최종 고정 차수(Fixed Out-degree)
        build_algo="ivf_pq"             # 초기 kNN 그래프 구축 알고리즘(ivf_pq 또는 nn_descent)
    )
    
    # 2. 인덱스 구축 (GPU 상에서 연속 실행)
    index = cagra.build(build_params, X_pca_cp, resources=handle)
    build_time = time.time() - t0
    print(f"[CAGRA] Index Construction completed in {build_time:.2f}s.")
    
    # 3. kNN 질의(Query) 탐색 파라미터 구성
    t_search = time.time()
    search_params = cagra.SearchParams(
        max_queries=100_000,            # 워프 스트리밍 배치 크기
        itopk_size=64                   # 내부 우선순위 큐 크기
    )
    
    # 4. 전체 100만 세포에 대한 배치 질의 실행
    # 질의=데이터셋이므로 각 세포의 이웃 목록에는 자기 자신이 포함된다 → k+1개를 질의한 뒤 자기 자신을 제거한다
    # (반환값은 CuPy가 아니라 device_ndarray이므로 cp.asarray로 변환; 기본 metric은 sqeuclidean이므로 거리는 제곱 거리)
    distances, neighbors = cagra.search(
        search_params, index, X_pca_cp, k + 1, resources=handle
    )
    nbr = cp.asarray(neighbors)
    dist = cp.sqrt(cp.maximum(cp.asarray(distances), 0.0))
    is_self = nbr == cp.arange(N, dtype=nbr.dtype)[:, None]
    is_self[:, -1] |= ~is_self.any(axis=1)      # 근사 탐색에서 자기 자신이 누락된 행은 가장 먼 이웃 1개를 제거
    keep = ~is_self
    knn_indices = nbr[keep].reshape(N, k)
    knn_distances = dist[keep].reshape(N, k)
    search_time = time.time() - t_search
    print(f"[CAGRA] kNN Search (k={k}) for 1M cells completed in {search_time:.2f}s.")
    print(f"[CAGRA] Total Graph Time: {time.time() - t0:.2f}s.")
    
    return knn_indices, knn_distances


def convert_knn_to_cugraph(knn_indices: cp.ndarray, knn_distances: cp.ndarray) -> cugraph.Graph:
    """
    kNN 탐색 결과를 cuGraph Graph 객체(CSR / Edge List)로 변환한다.
    가중치는 UMAP 스타일 퍼지 지수 감쇠(Fuzzy Simplicial Set Kernel)를 적용.
    """
    t0 = time.time()
    N, k = knn_indices.shape
    print(f"[Graph Prep] Converting kNN tensors into cuGraph Edge List (Total Edges: {N*k:,})...")
    
    # 1. Source 및 Destination 노드 벡터 구성
    src_nodes = cp.repeat(cp.arange(N, dtype=cp.int32), k)
    dst_nodes = knn_indices.ravel().astype(cp.int32)
    
    # 2. 거리 기반 지수 가중치 계산: w_ij = exp(-d_ij)
    weights = cp.exp(-cp.maximum(knn_distances.ravel() - 1e-3, 0.0)).astype(cp.float32)
    
    # 3. cuDF GPU DataFrame 생성 (Zero-Copy)
    edge_df = cudf.DataFrame({
        "source": src_nodes,
        "destination": dst_nodes,
        "weight": weights
    })
    
    # 4. 무방향 가중치 그래프 인스턴스화
    G = cugraph.Graph(directed=False)
    G.from_cudf_edgelist(
        edge_df, source="source", destination="destination", edge_attr="weight"
    )
    print(f"[Graph Prep] cuGraph initialized in {time.time() - t0:.2f}s. Number of Vertices: {G.number_of_vertices():,}")
    return G


def run_cugraph_leiden_clustering(G: cugraph.Graph, resolution: float = 0.8, 
                                  max_iter: int = 50) -> cudf.DataFrame:
    """
    cuGraph 병렬 Leiden 알고리즘을 실행하여 100만 세포의 커뮤니티 분할을 수행한다.
    """
    print(f"\n[Leiden] Starting cuGraph Parallel Leiden (Resolution: {resolution}, Max Iter: {max_iter})...")
    t0 = time.time()
    
    # GPU 병렬 Leiden 커널 디스패치 (내부 충돌 회피 방식은 cuGraph 구현에 따름)
    parts, modularity_score = cugraph.leiden(
        G,
        resolution=resolution,
        max_iter=max_iter
    )
    
    elapsed = time.time() - t0
    n_clusters = parts["partition"].nunique()
    print("="*65)
    print(f"[Leiden Success] Clustered {G.number_of_vertices():,} cells into {n_clusters} communities!")
    print(f"Final Graph Modularity (Q) : {modularity_score:.5f}")
    print(f"Total Leiden Runtime       : {elapsed:.3f} seconds")
    print("="*65 + "\n")
    
    return parts
'''
POST = '\nimport cupy as cp\ncp.random.seed(0)\nn = 25000\nX = cp.random.standard_normal((n, 50), dtype=cp.float32)      # HARNESS synthetic PCs\nidx, dist = ns["build_cagra_knn_graph_1m"](X, k=15)\nassert idx.shape == (n, 15) and dist.shape == (n, 15)\nG = ns["convert_knn_to_cugraph"](idx, dist)\nparts = ns["run_cugraph_leiden_clustering"](G, resolution=0.8, max_iter=50)\nassert len(parts) == n\nprint("HARNESS: leiden returned", len(parts), "rows")\n'
run_block('B5_25k', 'tracks/08_large_scale_sparse/docs/08_GPU_GRAPH_COMMUNITY_DETECTION.md:171', B5_SRC, POST, notes='', replace=[])

### B5_100k  `tracks/08_large_scale_sparse/docs/08_GPU_GRAPH_COMMUNITY_DETECTION.md:171` (T08 ch08 CAGRA + cuGraph Leiden)
100,000 random 50-d points (the chapter targets 1M).

In [ ]:
B5_SRC = r'''"""
Large-Scale Single-Cell Graph Construction & Community Detection
NVIDIA CAGRA + cuGraph Leiden on 1,000,000 Prostate Cancer Atlas
"""

import time
import numpy as np
import cupy as cp
import cudf
import cugraph
# CAGRA는 RAFT(pylibraft)에서 RAPIDS cuVS로 이전되었다 (cuvs.neighbors.cagra; cuVS 소스의 시그니처 기준, GPU 미실행)
from cuvs.common import Resources
from cuvs.neighbors import cagra
import scipy.sparse as sp
import cupyx.scipy.sparse as cpsp
from typing import Tuple


def build_cagra_knn_graph_1m(X_pca_cp: cp.ndarray, k: int = 15) -> Tuple[cp.ndarray, cp.ndarray]:
    """
    NVIDIA CAGRA를 활용하여 100만 세포(50 PCs)의 kNN 그래프를 GPU 네이티브로 초고속 구축한다.
    
    Parameters:
        X_pca_cp: GPU VRAM에 상주하는 CuPy float32 배열 (1,000,000 x 50)
        k: 탐색할 최근접 이웃 수 (기본 15)
        
    Returns:
        knn_indices: 각 세포별 k개 이웃의 정점 인덱스 (1,000,000 x k, int32)
        knn_distances: 유클리드 거리 행렬 (1,000,000 x k, float32)
    """
    N, d = X_pca_cp.shape
    print(f"\n[CAGRA] Initializing Fixed Out-Degree Graph Index for {N:,} cells (dim={d})...")
    t0 = time.time()
    
    handle = Resources()
    
    # 1. CAGRA 인덱스 빌드 파라미터 구성
    build_params = cagra.IndexParams(
        intermediate_graph_degree=64,   # 중간 탐색 그래프 차수
        graph_degree=32,                # 최종 고정 차수(Fixed Out-degree)
        build_algo="ivf_pq"             # 초기 kNN 그래프 구축 알고리즘(ivf_pq 또는 nn_descent)
    )
    
    # 2. 인덱스 구축 (GPU 상에서 연속 실행)
    index = cagra.build(build_params, X_pca_cp, resources=handle)
    build_time = time.time() - t0
    print(f"[CAGRA] Index Construction completed in {build_time:.2f}s.")
    
    # 3. kNN 질의(Query) 탐색 파라미터 구성
    t_search = time.time()
    search_params = cagra.SearchParams(
        max_queries=100_000,            # 워프 스트리밍 배치 크기
        itopk_size=64                   # 내부 우선순위 큐 크기
    )
    
    # 4. 전체 100만 세포에 대한 배치 질의 실행
    # 질의=데이터셋이므로 각 세포의 이웃 목록에는 자기 자신이 포함된다 → k+1개를 질의한 뒤 자기 자신을 제거한다
    # (반환값은 CuPy가 아니라 device_ndarray이므로 cp.asarray로 변환; 기본 metric은 sqeuclidean이므로 거리는 제곱 거리)
    distances, neighbors = cagra.search(
        search_params, index, X_pca_cp, k + 1, resources=handle
    )
    nbr = cp.asarray(neighbors)
    dist = cp.sqrt(cp.maximum(cp.asarray(distances), 0.0))
    is_self = nbr == cp.arange(N, dtype=nbr.dtype)[:, None]
    is_self[:, -1] |= ~is_self.any(axis=1)      # 근사 탐색에서 자기 자신이 누락된 행은 가장 먼 이웃 1개를 제거
    keep = ~is_self
    knn_indices = nbr[keep].reshape(N, k)
    knn_distances = dist[keep].reshape(N, k)
    search_time = time.time() - t_search
    print(f"[CAGRA] kNN Search (k={k}) for 1M cells completed in {search_time:.2f}s.")
    print(f"[CAGRA] Total Graph Time: {time.time() - t0:.2f}s.")
    
    return knn_indices, knn_distances


def convert_knn_to_cugraph(knn_indices: cp.ndarray, knn_distances: cp.ndarray) -> cugraph.Graph:
    """
    kNN 탐색 결과를 cuGraph Graph 객체(CSR / Edge List)로 변환한다.
    가중치는 UMAP 스타일 퍼지 지수 감쇠(Fuzzy Simplicial Set Kernel)를 적용.
    """
    t0 = time.time()
    N, k = knn_indices.shape
    print(f"[Graph Prep] Converting kNN tensors into cuGraph Edge List (Total Edges: {N*k:,})...")
    
    # 1. Source 및 Destination 노드 벡터 구성
    src_nodes = cp.repeat(cp.arange(N, dtype=cp.int32), k)
    dst_nodes = knn_indices.ravel().astype(cp.int32)
    
    # 2. 거리 기반 지수 가중치 계산: w_ij = exp(-d_ij)
    weights = cp.exp(-cp.maximum(knn_distances.ravel() - 1e-3, 0.0)).astype(cp.float32)
    
    # 3. cuDF GPU DataFrame 생성 (Zero-Copy)
    edge_df = cudf.DataFrame({
        "source": src_nodes,
        "destination": dst_nodes,
        "weight": weights
    })
    
    # 4. 무방향 가중치 그래프 인스턴스화
    G = cugraph.Graph(directed=False)
    G.from_cudf_edgelist(
        edge_df, source="source", destination="destination", edge_attr="weight"
    )
    print(f"[Graph Prep] cuGraph initialized in {time.time() - t0:.2f}s. Number of Vertices: {G.number_of_vertices():,}")
    return G


def run_cugraph_leiden_clustering(G: cugraph.Graph, resolution: float = 0.8, 
                                  max_iter: int = 50) -> cudf.DataFrame:
    """
    cuGraph 병렬 Leiden 알고리즘을 실행하여 100만 세포의 커뮤니티 분할을 수행한다.
    """
    print(f"\n[Leiden] Starting cuGraph Parallel Leiden (Resolution: {resolution}, Max Iter: {max_iter})...")
    t0 = time.time()
    
    # GPU 병렬 Leiden 커널 디스패치 (내부 충돌 회피 방식은 cuGraph 구현에 따름)
    parts, modularity_score = cugraph.leiden(
        G,
        resolution=resolution,
        max_iter=max_iter
    )
    
    elapsed = time.time() - t0
    n_clusters = parts["partition"].nunique()
    print("="*65)
    print(f"[Leiden Success] Clustered {G.number_of_vertices():,} cells into {n_clusters} communities!")
    print(f"Final Graph Modularity (Q) : {modularity_score:.5f}")
    print(f"Total Leiden Runtime       : {elapsed:.3f} seconds")
    print("="*65 + "\n")
    
    return parts
'''
POST = '\nimport cupy as cp\ncp.random.seed(0)\nn = 100000\nX = cp.random.standard_normal((n, 50), dtype=cp.float32)      # HARNESS synthetic PCs\nidx, dist = ns["build_cagra_knn_graph_1m"](X, k=15)\nassert idx.shape == (n, 15) and dist.shape == (n, 15)\nG = ns["convert_knn_to_cugraph"](idx, dist)\nparts = ns["run_cugraph_leiden_clustering"](G, resolution=0.8, max_iter=50)\nassert len(parts) == n\nprint("HARNESS: leiden returned", len(parts), "rows")\n'
run_block('B5_100k', 'tracks/08_large_scale_sparse/docs/08_GPU_GRAPH_COMMUNITY_DETECTION.md:171', B5_SRC, POST, notes='', replace=[])

### B7_100k  `tracks/08_large_scale_sparse/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:205` (T08 ch09 memory guard + cuML PCA)
100,000 x 3,000 float32; guard thresholds are the chapter's (70/65 GB), so the defragmentation branch is expected NOT to trigger.

In [ ]:
B7_SRC = r'''"""
Enterprise Single-Cell Pipeline Diagnostic & Memory Guard Module
Real-time VRAM/RAM Monitoring and Automatic Fragmentation Defense
"""

import functools
import gc
import time
import os
import psutil
import cupy as cp
import rmm


def single_cell_memory_guard(stage_name: str, warn_vram_gb: float = 70.0):
    """
    단일세포 파이프라인의 각 단계 전후로 CPU RAM과 GPU VRAM의 누수 및 단편화를 정밀 추적하고
    비정상적인 메모리 잔류를 강제 회수하는 방어적 데코레이터.
    """
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            # 1. 실행 전 메모리 상태 스냅샷
            process = psutil.Process(os.getpid())
            ram_before_mb = process.memory_info().rss / (1024 * 1024)
            
            # GPU VRAM 상태 쿼리
            free_bytes, total_bytes = cp.cuda.runtime.memGetInfo()
            vram_used_before_gb = (total_bytes - free_bytes) / (1024**3)
            total_vram_gb = total_bytes / (1024**3)

            print(f"\n[MemoryGuard: START] >>> Stage: {stage_name}")
            print(f"  CPU RAM Baseline : {ram_before_mb:.2f} MB")
            print(f"  GPU VRAM Used    : {vram_used_before_gb:.2f} GB / {total_vram_gb:.2f} GB")

            t0 = time.time()
            try:
                result = func(*args, **kwargs)
            except cp.cuda.memory.OutOfMemoryError as oom:
                print(f"\n[CRITICAL OOM DETECTED] Stage '{stage_name}' crashed with CUDA Out of Memory!")
                print("[Emergency Recovery] Flushing CuPy Memory Pool and forcing Python GC...")
                # 긴급 메모리 회수
                gc.collect()
                cp.get_default_memory_pool().free_all_blocks()
                raise oom

            elapsed = time.time() - t0

            # 2. 실행 후 메모리 상태 스냅샷
            ram_after_mb = process.memory_info().rss / (1024 * 1024)
            free_bytes, _ = cp.cuda.runtime.memGetInfo()
            vram_used_after_gb = (total_bytes - free_bytes) / (1024**3)
            vram_delta_gb = vram_used_after_gb - vram_used_before_gb

            print(f"[MemoryGuard: END] <<< Stage: {stage_name} (Duration: {elapsed:.2f}s)")
            print(f"  CPU RAM Delta    : {ram_after_mb - ram_before_mb:+.2f} MB (Current: {ram_after_mb:.2f} MB)")
            print(f"  GPU VRAM Delta   : {vram_delta_gb:+.2f} GB (Current: {vram_used_after_gb:.2f} GB)")

            # 3. 위험 임계치 경고 및 방어 조치
            if vram_used_after_gb >= warn_vram_gb:
                print(f"  [WARNING] VRAM utilization exceeds {warn_vram_gb} GB! Performing defensive defragmentation...")
                gc.collect()
                cp.get_default_memory_pool().free_all_blocks()
                free_bytes_post, _ = cp.cuda.runtime.memGetInfo()
                reclaimed_gb = ((total_bytes - free_bytes_post) - vram_used_after_gb)
                print(f"  [Recovery Complete] Reclaimed {abs(reclaimed_gb):.2f} GB of fragmented VRAM.")

            return result
        return wrapper
    return decorator


# 사용 예시
@single_cell_memory_guard(stage_name="cuML PCA 50 PCs", warn_vram_gb=65.0)
def execute_safe_pca(X_gpu, n_components=50):
    import cuml
    # cuML PCA의 svd_solver는 'full' | 'jacobi' | 'auto' 만 지원한다 ('randomized'는 없음; 랜덤화 SVD는 rsc.pp.pca(..., svd_solver="randomized") 사용)
    pca = cuml.PCA(n_components=n_components, svd_solver="jacobi")
    return pca.fit_transform(X_gpu)
'''
POST = '\nimport cupy as cp\nn = 100000\nX = cp.random.standard_normal((n, 3000), dtype=cp.float32)    # HARNESS synthetic matrix (3000 HVGs)\nout = ns["execute_safe_pca"](X, n_components=50)\nassert out.shape == (n, 50)\nprint("HARNESS: PCA output", out.shape)\n'
run_block('B7_100k', 'tracks/08_large_scale_sparse/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:205', B7_SRC, POST, notes='', replace=[])

### B6  `tracks/08_large_scale_sparse/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:105` (T08 ch09 RMM managed memory)
Verbatim `initial_pool_size="64GB"` (run last: RMM re-initialisation). Plus a 4 GiB allocation and an informational 90 GB oversubscription probe.

In [ ]:
B6_SRC = r'''import rmm

# VRAM 80GB 초과 시 시스템 DDR5 RAM으로 자동 투명 페이징(Paging)
rmm.reinitialize(
    managed_memory=True,  # cudaMallocManaged 활성화
    pool_allocator=True,
    initial_pool_size="64GB"  # 문자열은 RMM이 직접 해석 (GB=10^9 bytes)
)
'''
POST = '\nimport cupy as cp\nfrom rmm.allocators.cupy import rmm_cupy_allocator\ncp.cuda.set_allocator(rmm_cupy_allocator)\nx = cp.ones(4 * 1024**3 // 4, dtype=cp.float32)               # HARNESS: 4 GiB inside the 64GB managed pool\ncp.cuda.runtime.deviceSynchronize()\nprint("HARNESS: 4 GiB allocation inside the managed pool OK")\ndel x\ntry:                                                          # HARNESS probe of the chapter claim "exceeding physical 80GB VRAM"\n    y = cp.empty(90 * 10**9 // 4, dtype=cp.float32)\n    print("HARNESS PROBE: 90 GB managed allocation SUCCEEDED")\n    del y\nexcept Exception as e:\n    print("HARNESS PROBE: 90 GB managed allocation FAILED:", type(e).__name__, str(e)[:200])\n'
run_block('B6', 'tracks/08_large_scale_sparse/docs/09_TROUBLESHOOTING_AND_BEST_PRACTICES.md:105', B6_SRC, POST, notes='', replace=[])

## 5. Write results

In [ ]:
import json
RESULTS["meta"] = {"date": time.strftime("%Y-%m-%d %H:%M:%S"), "gpu": GPU_NAME, "gpu_mem_bytes": GPU_MEM_BYTES,
                   "host": HOST, "versions": VERSIONS, "atlas_nnz": {str(k): v for k, v in NNZ.items()}}
json.dump(RESULTS, open("gpu_results_80gb.json", "w"), indent=2, ensure_ascii=False)
print(f"{'block':9s} {'status':8s} {'sec':>8s} {'peakGPU MiB':>12s} {'peakRSS MiB':>12s}")
for k, r in RESULTS["blocks"].items():
    print(f"{k:9s} {r['status']:8s} {str(r.get('elapsed_s')):>8s} {str(r.get('peak_gpu_used_mib')):>12s} {str(r.get('peak_host_rss_mib')):>12s}")
print(sum(r["status"] == "PASS" for r in RESULTS["blocks"].values()), "/", len(RESULTS["blocks"]), "PASS; saved gpu_results_80gb.json")